# Redes Convolucionales Modernas

In [1]:
%matplotlib inline
import torch
import torchvision
from torch import nn
import torchvision.transforms as T
from torch.utils import data
from torch.nn import functional as F
import matplotlib.pyplot as plt


def init_cnn(module):
    if type(module) == nn.Linear or type(module) == nn.Conv2d:
        nn.init.xavier_uniform_(module.weight)


In [2]:
def load_data_fashion_mnist(batch_size, resize=None):
    trans = [T.ToTensor()]
    if resize:
        trans.insert(0, T.Resize(resize))
    trans = T.Compose(trans)
    mnist_train = torchvision.datasets.FashionMNIST(
        root="../data", train=True, transform=trans, download=True)
    mnist_test = torchvision.datasets.FashionMNIST(
        root="../data", train=False, transform=trans, download=True)
    return (data.DataLoader(mnist_train, batch_size, shuffle=True,
                            num_workers=1),
            data.DataLoader(mnist_test, batch_size, shuffle=False,
                            num_workers=1))

def accuracy(y_hat, y):
    """Compute the number of correct predictions."""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1)
    cmp = y_hat.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum())


In [3]:
def train_FashionMNIST_classifier(model, lr, num_epochs, resize=None):
  batch_size= 128
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model = model.to(device)
  loss = nn.CrossEntropyLoss(reduction='none')
  trainer = torch.optim.Adam(model.parameters())
  train_iter, test_iter = load_data_fashion_mnist(batch_size,resize=resize)

  for epoch in range(num_epochs):
      L = 0.0
      N = 0
      Acc = 0.0
      TestAcc = 0.0
      TestN = 0
      for X, y in train_iter:
          X, y = X.to(device), y.to(device)
          l = loss(model(X),y)
          trainer.zero_grad()
          l.mean().backward()
          trainer.step()
          L += l.sum()
          N += l.numel()
          Acc += accuracy(model(X), y)
      for X, y in test_iter:
          X, y = X.to(device), y.to(device)
          TestN += y.numel()
          TestAcc += accuracy(model(X), y)
      print(f'epoch {epoch + 1}, loss {(L/N):f}\
            , train accuracy  {(Acc/N):f}, test accuracy {(TestAcc/TestN):f}')

# Redes con Conexiones Residuales (ResNet)

A medida que diseñamos redes cada vez más profundas, se vuelve imperativo comprender cómo agregar capas puede aumentar la complejidad y la expresividad de la red. Aún más importante es la capacidad de diseñar redes donde agregar capas hace que las redes sean estrictamente más expresivas en lugar de simplemente diferentes. Para hacer algún progreso necesitamos un poco de matemáticas.

## Familias de funciones



Considere $\mathcal{F}$, la familia de funciones que puede alcanzar una arquitectura de red específica (junto con las tasas de aprendizaje y otras configuraciones de hiperparámetros). Es decir, para todo $f \in \mathcal{F}$ existe algún conjunto de parámetros (p. ej., pesos y sesgos) que se pueden obtener a través del entrenamiento en un conjunto de datos adecuado. Supongamos que $f^*$ es la función "de verdad" que realmente nos gustaría encontrar. Si está dentro de $\mathcal{F}$, vamos a poder encontrarla, pero por lo general no tendremos tanta suerte. En su lugar, intentaremos encontrar una $f^*_\mathcal{F}$, que es nuestra mejor apuesta dentro de $\mathcal{F}$. Por ejemplo, dado un conjunto de datos con características $\mathbf{X}$ y etiquetas $\mathbf{y}$, podríamos intentar encontrarlo resolviendo el siguiente problema de optimización:

$$f^*_\mathcal{F} \stackrel{\mathrm{def}}{=} \mathop{\mathrm{argmin}}_f L(\mathbf{X}, \mathbf{y}, f) \text{ sujeto a } f \in \mathcal{F}.$$

Sabemos que la regularización puede controlar la complejidad de $\mathcal{F}$ y lograr consistencia, por lo que un mayor tamaño de datos de entrenamiento generalmente conduce a mejores $f^*_\mathcal{F}$. Es razonable suponer que si diseñamos una arquitectura diferente y más poderosa $\mathcal{F}'$ deberíamos llegar a un mejor resultado. En otras palabras, esperaríamos que $f^*_{\mathcal{F}'}$ sea "mejor" que $f^*_{\mathcal{F}}$. Sin embargo, si $\mathcal{F} \not\subseteq \mathcal{F}'$ no hay garantía de que esto suceda. De hecho, $f^*_{\mathcal{F}'}$ podría ser peor. Como se ilustra en la figura, para las familias de funciones no anidadas, una clase de función más grande no siempre se acerca a la función de "verdad" $f^*$. Por ejemplo,
a la izquierda de la figura, aunque $\mathcal{F}_3$ está más cerca de $f^*$ que $\mathcal{F}_1$, $\mathcal{F}_6$ se aleja y no hay garantía de que aumentar aún más la complejidad puede reducir la distancia de $f^*$. Con clases de funciones anidadas donde $\mathcal{F}_1 \subseteq \ldots \subseteq \mathcal{F}_6$ como a la derecha de la figura, podemos evitar el problema antes mencionado de las familias de funciones no anidadas.

![Imgur](https://i.imgur.com/LMVC1zT.png)



Por lo tanto, solo si las familias de funciones más grandes contienen a las más pequeñas, se garantiza que al aumentarlas, aumenta estrictamente el poder expresivo de la red. Para redes neuronales profundas, si podemos entrenar la capa recién agregada en una función de identidad $f(\mathbf{x}) = \mathbf{x}$, el nuevo modelo será tan efectivo como el modelo original.

En el corazón de **ResNet**, la **red residual** propuesta en 2015, está la idea de que cada capa adicional debería contener más fácilmente la función de identidad como uno de sus elementos. Estas consideraciones son bastante profundas pero llevaron a una solución sorprendentemente simple, un ***bloque residual***. Con él, ResNet ganó el Desafío de ImageNet en 2015. El diseño tuvo una profunda influencia en cómo construir redes neuronales profundas. Por ejemplo, los transformers los utilizan para apilar muchas capas de redes de manera eficiente. También se utiliza en redes neuronales basadas en grafos y, como concepto básico, se ha utilizado ampliamente en visión artificial.

## Bloques Residuales

Centrémonos en una parte local de una red neuronal, como se muestra en la figura. Sea la entrada denotada por $\mathbf{x}$, supongamos que el mapeo subyacente deseado que queremos obtener mediante el aprendizaje es $f(\mathbf{x})$, que se usará como entrada para la función de activación en la parte superior.

En la parte izquierda, la parte dentro del cuadro de línea punteada
debe aprender directamente el mapeo $f(\mathbf{x})$. En cambio, en la de la derecha, la porción dentro del cuadro de línea punteada necesita aprender el ***mapeo residual*** $g(\mathbf{x}) = f(\mathbf{x}) - \mathbf{x}$, que es de donde el bloque residual deriva su nombre. Si el mapeo de identidad $f(\mathbf{x}) = \mathbf{x}$ es el mapeo subyacente deseado,
el mapeo residual se establece a $g(\mathbf{x}) = 0$ y, por lo tanto, es más fácil de aprender: solo necesitamos empujar los pesos y sesgos
de la capa superior dentro del cuadro de línea de puntos a cero.

La figura de la derecha ilustra el ***bloque residual*** de ResNet, donde la línea continua que lleva la entrada de capa $\mathbf{x}$ al operador de suma se denomina *conexión residual* **texto en negrita**(o *conexión de acceso directo*).
Con bloques residuales, las entradas pueden propagarse más rápido a través de las conexiones residuales entre capas. De hecho, el bloque residual se puede considerar como un caso especial del bloque Inception de múltiples ramas: tiene dos ramas, una de las cuales es el mapeo de identidad.

![Imgur](https://i.imgur.com/Uh2qg4W.png)

ResNet sigue el diseño completo de capa convolucional 3×3 de VGG. El bloque residual tiene dos capas convolucionales de 3×3 con el mismo número de canales de salida. A cada capa convolucional le sigue una capa de normalización por lotes y una función de activación de ReLU. Luego, nos saltamos estas dos operaciones de convolución y agregamos la entrada directamente antes de la función de activación ReLU final. Este tipo de diseño requiere que la salida de las dos capas convolucionales tengan la misma forma que la entrada, para que puedan sumarse. Si queremos cambiar la cantidad de canales, debemos introducir una capa convolucional adicional de 1 × 1 para transformar la entrada en la forma deseada para la operación de suma.

![Imgur](https://i.imgur.com/NIzaIdk.png)



Echemos un vistazo al código de abajo.

In [4]:
class ResidualBlock(nn.Module):
    """The Residual block of ResNet."""
    def __init__(self, num_channels, use_1x1conv=False, strides=1):
        super().__init__()
        self.conv1 = nn.LazyConv2d(num_channels, kernel_size=3, padding=1,
                                   stride=strides)
        self.conv2 = nn.LazyConv2d(num_channels, kernel_size=3, padding=1)
        if use_1x1conv:
            self.conv3 = nn.LazyConv2d(num_channels, kernel_size=1,
                                       stride=strides)
        else:
            self.conv3 = None
        self.bn1 = nn.LazyBatchNorm2d()
        self.bn2 = nn.LazyBatchNorm2d()

    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:
            X = self.conv3(X)
        Y += X
        return F.relu(Y)

Ahora veamos una situación en la que la entrada y la salida tienen la misma forma, donde no se necesita la convolución de $1 \times 1$.


In [5]:
blk = ResidualBlock(3)
X = torch.randn(4, 3, 6, 6)
blk(X).shape

torch.Size([4, 3, 6, 6])

También tenemos la opción de reducir a la mitad la altura y el ancho de salida mientras aumentamos el número de canales de salida. En este caso usamos convoluciones de $1 \times 1$ a través de `use_1x1conv=True`. Esto resulta útil al principio de cada bloque ResNet para reducir la dimensionalidad espacial con`strides=2`.


In [6]:
blk = ResidualBlock(6, use_1x1conv=True, strides=2)
blk(X).shape

torch.Size([4, 6, 3, 3])

## **ResNet**


![Imgur](https://i.imgur.com/Wx0EJoF.png?1)

Las dos primeras capas de ResNet son las mismas que las de GoogLeNet: la capa convolucional de $7\times 7$ con 64 canales de salida y un stride de 2 es seguida por la capa de max-pooling de $3\times 3$ con un stride de 2. La diferencia es la capa de batch normalization agregada después de cada capa convolucional en ResNet.


In [7]:
class Base(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.LazyConv2d(64, kernel_size=7, stride=2, padding=3),
            nn.LazyBatchNorm2d(), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

    def forward(self, X):
        return self.net(X)

GoogLeNet utiliza cuatro módulos compuestos por bloques Inception. Sin embargo, ResNet utiliza cuatro módulos formados por bloques residuales, cada uno de los cuales utiliza varios bloques residuales con el mismo número de canales de salida. El número de canales en el primer módulo es el mismo que el número de canales de entrada. Dado que ya se ha utilizado una capa de max_pooling con un stride de 2, no es necesario reducir la altura y el ancho. En el primer bloque residual para cada uno de los módulos subsiguientes, se duplica el número de canales con respecto al del módulo anterior, y se reducen a la mitad la altura y la anchura.


In [8]:
class ResModule(nn.Module):
    def __init__(self, num_residuals, num_channels, first_module=False):
        super().__init__()
        blk = []
        for i in range(num_residuals):
            if i == 0 and not first_module:
                blk.append(ResidualBlock(num_channels, use_1x1conv=True, strides=2))
            else:
                blk.append(ResidualBlock(num_channels))
        self.net = nn.Sequential(*blk)

    def forward(self, X):
        return self.net(X)


Luego, agregamos todos los módulos a ResNet. Aquí, se utilizan dos bloques residuales para cada módulo. Por último, al igual que GoogLeNet, agregamos una capa de avg-pooling global, seguida de la salida de la capa densa.


In [9]:
class ResNet(nn.Module):
    def __init__(self, arch, lr=0.1, num_classes=10):
        super(ResNet, self).__init__()
        self.net = nn.Sequential(Base())
        for i, b in enumerate(arch):
            self.net.add_module(f'b{i+2}', ResModule(*b, first_module=(i==0)))
        self.net.add_module('last', nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.LazyLinear(num_classes)))
        self.net.apply(init_cnn)

    def forward(self, X):
        return self.net(X)

In [10]:
class ResNet18(ResNet):
    def __init__(self, lr=0.1, num_classes=10):
        super().__init__(((2, 64), (2, 128), (2, 256), (2, 512)),
                       lr, num_classes)

    def forward(self, X):
        return self.net(X)

Hay 4 capas convolucionales en cada módulo (excluyendo la capa convolucional $1\times 1$). Junto con la primera capa convolucional $7\times 7$ y la última capa densa, hay 18 capas en total. Por lo tanto, este modelo se conoce comúnmente como ResNet-18. Al configurar diferentes números de canales y bloques residuales en el módulo, podemos crear diferentes modelos ResNet, como el  más profundo ResNet-152 de 152 capas. Aunque la arquitectura principal de ResNet es similar a la de GoogLeNet, la estructura de ResNet es más simple y fácil de modificar. Todos estos factores han resultado en el uso rápido y generalizado de ResNet.

Antes de entrenar a ResNet, observemos cómo cambia la forma de entrada en diferentes módulos en ResNet. Como en todas las arquitecturas anteriores, la resolución disminuye mientras que la cantidad de canales aumenta hasta el punto en que una capa de avg-pooling global agrega todas las features.


In [11]:
res_net_18 = ResNet18()
print(res_net_18)

ResNet18(
  (net): Sequential(
    (0): Base(
      (net): Sequential(
        (0): LazyConv2d(0, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
        (1): LazyBatchNorm2d(0, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      )
    )
    (b2): ResModule(
      (net): Sequential(
        (0): ResidualBlock(
          (conv1): LazyConv2d(0, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (conv2): LazyConv2d(0, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (bn1): LazyBatchNorm2d(0, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (bn2): LazyBatchNorm2d(0, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): ResidualBlock(
          (conv1): LazyConv2d(0, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (conv2): LazyConv2d(0, 64, kernel_size=(3, 

## Entrenamiento

Entrenamos a ResNet en el conjunto de datos Fashion-MNIST, como antes. ResNet es una arquitectura bastante poderosa y flexible.



In [12]:
#Con T4, 2.5 minutos
lr, num_epochs = 0.01, 5
train_FashionMNIST_classifier(res_net_18,lr,num_epochs,resize=(50, 50))

100%|██████████| 26.4M/26.4M [00:00<00:00, 115MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 3.77MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 62.8MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 14.6MB/s]


epoch 1, loss 0.388854            , train accuracy  0.879383, test accuracy 0.891900
epoch 2, loss 0.260985            , train accuracy  0.918883, test accuracy 0.904400
epoch 3, loss 0.222369            , train accuracy  0.931667, test accuracy 0.903900
epoch 4, loss 0.197595            , train accuracy  0.940167, test accuracy 0.915000
epoch 5, loss 0.177042            , train accuracy  0.946950, test accuracy 0.920600


## ResNeXt





Uno de los desafíos que uno encuentra en el diseño de ResNet es el equilibrio entre la no linealidad y la dimensionalidad dentro de un bloque determinado. Es decir, podríamos agregar más no linealidad aumentando el número de capas o aumentando el ancho de las convoluciones. Una estrategia alternativa es aumentar el número de canales que pueden transportar información entre bloques. Desafortunadamente, esto último viene con una penalización cuadrática ya que el costo computacional de ingerir $c_i$ canales y emitir $c_o$ canales es proporcional a $\mathcal{O}(c_i \cdot c_o)$.

Podemos inspirarnos en el bloque Inception de GoogLeNet, que tiene información que fluye a través del bloque en grupos separados, para aplicar la idea de múltiples grupos independientes al bloque ResNet. Esto fue lo que condujo al diseño de ResNeXt en 2017. A diferencia de la mezcla heterogénea de transformaciones en Inception, ResNeXt adopta la ***misma*** transformación en todas las ramas, minimizando así la necesidad de tuneo manual de cada rama.

![Imgur](https://i.imgur.com/Znx5KrJ.png)

Dividir una convolución de $c_i$ a $c_o$ canales en una de $g$ grupos de tamaño $c_i/g$ generando $g$ salidas de tamaño $c_o/g$ se llama, muy apropiadamente, una ***convolución agrupada*** . El costo computacional (proporcionalmente) se reduce de $\mathcal{O}(c_i \cdot c_o)$ a $\mathcal{O}(g \cdot (c_i/g) \cdot (c_o/g)) = \mathcal{ O}(c_i \cdot c_o / g)$, es decir, **es $g$ veces más rápido**. Aún mejor, la cantidad de parámetros necesarios para generar la salida también se reduce de una matriz $c_i \times c_o$ a $g$ matrices más pequeñas de tamaño $(c_i/g) \times (c_o/g)$, nuevamente **una reducción de $g$ veces**. En lo que sigue asumimos que tanto $c_i$ como $c_o$ son divisibles por $g$.

El único desafío en este diseño es que no se intercambia información entre los $g$ grupos. El bloque ResNeXt de la figura corrige esto de dos maneras: la convolución agrupada con un kernel de $3 \times 3$ se intercala entre dos convoluciones de $1 \times 1$. El segundo cumple una doble función al volver a cambiar el número de canales. El beneficio es que solo pagamos el costo de $\mathcal{O}(c \cdot b)$ por $1 \times 1$ kernels y podemos arreglárnoslas con un costo de $\mathcal{O}(b^2 / g)$ por $3 \times 3$ núcleos. Similar a la implementación del bloque residual en
ResNet, la conexión residual se puede reemplazar por una convolución de $1 \times 1$.

La figura de la derecha proporciona un resumen mucho más conciso del bloque de red resultante.

La siguiente implementación de la clase `ResNeXtBlock` toma como argumento `groups` ($g$), con `bot_channels` ($b$) canales intermedios (cuello de botella). Por último, cuando necesitamos reducir la altura y el ancho de la representación, agregamos un stride de $2$ configurando `use_1x1conv=True, strides=2`.

In [13]:
class ResNeXtBlock(nn.Module):
    """The ResNeXt block."""
    def __init__(self, num_channels, groups, bot_mul, use_1x1conv=False,
                 strides=1):
        super().__init__()
        bot_channels = int(round(num_channels * bot_mul))
        self.conv1 = nn.LazyConv2d(bot_channels, kernel_size=1, stride=1)
        self.conv2 = nn.LazyConv2d(bot_channels, kernel_size=3,
                                   stride=strides, padding=1,
                                   groups=bot_channels//groups)
        self.conv3 = nn.LazyConv2d(num_channels, kernel_size=1, stride=1)
        self.bn1 = nn.LazyBatchNorm2d()
        self.bn2 = nn.LazyBatchNorm2d()
        self.bn3 = nn.LazyBatchNorm2d()
        if use_1x1conv:
            self.conv4 = nn.LazyConv2d(num_channels, kernel_size=1,
                                       stride=strides)
            self.bn4 = nn.LazyBatchNorm2d()
        else:
            self.conv4 = None

    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = F.relu(self.bn2(self.conv2(Y)))
        Y = self.bn3(self.conv3(Y))
        if self.conv4:
            X = self.bn4(self.conv4(X))
        return F.relu(Y + X)

Su uso es completamente análogo al de `ResNetBlock` discutido anteriormente. Por ejemplo, cuando se usa (`use_1x1conv=False, strides=1`), la entrada y la salida tienen la misma forma. Alternativamente, configurar `use_1x1conv=True, strides=2` reduce a la mitad la altura y el ancho de salida.


In [14]:
blk = ResNeXtBlock(32, 16, 1)
X = torch.randn(4, 32, 96, 96)
blk(X).shape

torch.Size([4, 32, 96, 96])

# Redes Densamente Conectadas

ResNet cambió significativamente la visión de cómo parametrizar las funciones en las redes profundas. DenseNet (red convolucional densa) es, hasta cierto punto, la extensión lógica de este. DenseNet se caracteriza tanto por el patrón de **conectividad** (en el que cada capa se conecta a todas las capas anteriores) como por la operación de **concatenación** (en lugar del operador de suma en ResNet) para preservar y reutilizar características de capas anteriores. Para entender cómo llegar a él, tomemos un pequeño desvío a las matemáticas.

## De ResNet a DenseNet

Recuerde la expansión de Taylor para funciones. Para el punto $x = 0$ se puede escribir como

$$f(x) = f(0) + x \cdot \left[f'(0) + x \cdot \left[\frac{f''(0)}{2!} + x \cdot \left [\frac{f'''(0)}{3!} + \ldots \right]\right]\right].$$


El punto clave es que descompone una función en términos de orden cada vez más alto. De manera similar, ResNet descompone funciones en

$$f(\mathbf{x}) = \mathbf{x} + g(\mathbf{x}).$$

Es decir, ResNet descompone $f$ en un término lineal simple y uno no lineal más complejo. ¿Qué pasaría si quisiéramos capturar (no necesariamente agregar) información más allá de dos términos? Una de esas soluciones es DenseNet

![Imgur](https://i.imgur.com/Z8sw0Xp.png)

Como se muestra en la figura, la diferencia clave entre ResNet y DenseNet es que, en el último caso, las salidas se *concatenan* en lugar de agregarse. Como resultado, realizamos un mapeo de $\mathbf{x}$ a sus valores después de aplicar una secuencia de funciones cada vez más compleja:

$$\mathbf{x} \to \left[
\mathbf{x},
f_1(\mathbf{x}),
f_2\left(\left[\mathbf{x}, f_1\left(\mathbf{x}\right)\right]\right), f_3\left(\left[\mathbf{x}, f_1\left(\mathbf{x}\right), f_2\left(\left[\mathbf{x}, f_1\left(\mathbf{x}\right)\right]\right)\right]\right), \ldots\right].$$

Al final, todas estas funciones se combinan en MLP para reducir nuevamente la cantidad de características. En términos de implementación, esto es bastante simple: en lugar de agregar términos, los concatenamos. El nombre DenseNet surge del hecho de que el gráfico de dependencia entre variables se vuelve bastante denso. La última capa de tal cadena está densamente conectada a todas las capas anteriores.

Los componentes principales que componen una DenseNet son *bloques densos* y *capas de transición*. Los primeros definen cómo se concatenan las entradas y salidas, mientras que los segundos controlan el número de canales para que no sea demasiado grande, ya que la expansión  $\mathbf{x} \to \left[\mathbf{x}, f_1(\mathbf{x}), f_2\left(\left[\mathbf{x}, f_1\left(\mathbf{x}\right)\right]\right), \ldots \right]$  puede tener dimensiones bastante altas.


## Bloques densos








Un *bloque denso* consta de varios bloques de convolución, cada uno de los cuales utiliza el mismo número de canales de salida. En la propagación directa, sin embargo, concatenamos la entrada y la salida de cada bloque de convolución en la dimensión del canal.

![](https://i.imgur.com/wwy3lR1.gif)

DenseNet utiliza la estructura modificada de "normalización por lotes, activación y convolución" de ResNet. Primero, implementamos esta estructura de bloque de convolución.

![](https://i.imgur.com/onupQ8B.gif)


In [15]:
class Conv_Block(nn.Module):
    def __init__(self, num_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.LazyBatchNorm2d(), nn.ReLU(),
            nn.LazyConv2d(num_channels, kernel_size=3, padding=1))

    def forward(self, X):
        return self.net(X)


In [17]:
class DenseBlock(nn.Module):
    def __init__(self, num_convs, num_channels):
        super(DenseBlock, self).__init__()
        layer = []
        for i in range(num_convs):
            layer.append(Conv_Block(num_channels))
        self.net = nn.Sequential(*layer)

    def forward(self, X):
        for blk in self.net:
            Y = blk(X)
            X = torch.cat((X, Y), dim=1)
        return X

En el siguiente ejemplo, definimos una instancia `DenseBlock` con 2 bloques de convolución de 10 canales de salida. Al usar una entrada con 3 canales, obtendremos una salida con $3 + 10 + 10=23$ canales. El número de canales de bloque de convolución controla el crecimiento del número de canales de salida en relación con el número de canales de entrada. Esto también se conoce como la ***tasa de crecimiento***.




In [18]:
blk = DenseBlock(2, 10)
X = torch.randn(4, 3, 8, 8)
Y = blk(X)
Y.shape

torch.Size([4, 23, 8, 8])

Dado que cada capa recibe mapas de activación de todas las capas anteriores, la red puede ser más delgada y compacta, es decir, el número de canales puede ser menor. Por lo tanto, tiene una **mayor eficiencia computacional y eficiencia de memoria**.

![Imgur](https://i.imgur.com/N9H4Ttr.png)

Además de la eficiencia computacional, el bloque denso también tiene otras ventajas:

1. Mejora el problema del desvanecimiento de los gradientes: al unir a todas las capas con la salida del bloque a través de una conexión directa, el backpropagation puede modificar más facilmente las primeras capas. ![Imgur](https://i.imgur.com/qk8QQPe.png)

2. Permite una mayor **diversificación de las features** debido a que el clasificador usa features de todos los niveles de complejidad para tomar la decisión.

  <img src="https://i.imgur.com/BJyUSHa.png" width="660">

## Capas de transición

Dado que cada bloque denso aumentará el número de canales, agregar demasiados conducirá a un modelo excesivamente complejo. Se utiliza una ***capa de transición*** para controlar la complejidad del modelo. Reduce el número de canales usando una convolución $1\times 1$. Además, reduce a la mitad la altura y el ancho a través del avg_pooling con un stride de 2.
  <img src="https://i.imgur.com/NrBmAso.png" width="660">


In [19]:
class TransitionBlock(nn.Module):
    def __init__(self,num_channels):
        super().__init__()
        self.net = nn.Sequential(
          nn.LazyBatchNorm2d(), nn.ReLU(),
          nn.LazyConv2d(num_channels, kernel_size=1),
          nn.AvgPool2d(kernel_size=2, stride=2))

    def forward(self, X):
        return self.net(X)

Aplicamos una capa de transición con 10 canales a la salida del bloque denso en el ejemplo anterior. Esto reduce el número de canales de salida a 10 y reduce a la mitad el alto y el ancho.


In [20]:
print(Y.shape)
blk = TransitionBlock(10)
blk(Y).shape

torch.Size([4, 23, 8, 8])


torch.Size([4, 10, 4, 4])


## Modelo DenseNet



A continuación, construiremos un modelo DenseNet. DenseNet primero usa la misma base que ResNet (capa convolucional única y luego max-pool).

Luego, similar a los cuatro módulos compuestos por bloques residuales que usa ResNet, DenseNet usa cuatro bloques densos. Similar a ResNet, podemos establecer la cantidad de capas convolucionales utilizadas en cada bloque denso. Aquí lo configuramos en 4, de acuerdo con el modelo ResNet-18.  Además, establecemos el número de canales (es decir, la tasa de crecimiento) para las capas convolucionales en el bloque denso en 32, por lo que se agregarán 128 canales a cada bloque denso.

En ResNet, la altura y el ancho se reducen entre cada módulo por un bloque residual con un paso de 2. Aquí, usamos la capa de transición para reducir a la mitad la altura y el ancho y reducir a la mitad el número de canales. De manera similar a ResNet, una capa de avg-pooling global y una capa totalmente conectada se conectan al final para producir la salida.




In [21]:
class DenseNet(nn.Module):
    def __init__(self, num_channels=64, growth_rate=32, arch=(4, 4, 4, 4),
                lr=0.1, num_classes=10):
        super(DenseNet, self).__init__()
        # Base
        self.net = nn.Sequential(Base())
        # Cuerpo
        for i, num_convs in enumerate(arch):
            self.net.add_module(f'dense_blk{i+1}', DenseBlock(num_convs,
                                                              growth_rate))
            # El número de canales de salida en el bloque denso anterior
            num_channels += num_convs * growth_rate
            # Se agrega una capa de transición que reduce a la mitad
            # el número de canales entre los bloques densos
            if i != len(arch) - 1:
                num_channels //= 2
                self.net.add_module(f'tran_blk{i+1}',
                                    TransitionBlock(num_channels))
        #Cabeza
        self.net.add_module('last', nn.Sequential(
            nn.LazyBatchNorm2d(), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.LazyLinear(num_classes)))
        self.net.apply(init_cnn)

    def forward(self, X):
        return self.net(X)

![Imgur](https://i.imgur.com/NrBmAso.png)


## Entrenamiento

Dado que aquí estamos usando una red más profunda, en esta sección reduciremos la altura y el ancho de entrada de 224 a 96 para simplificar el cálculo.


In [22]:
# Con T4, 45 segundos para 1 epoch. Para 5 epoch son 3.75 min
dense_net = DenseNet()
lr, num_epochs = 0.01, 5
train_FashionMNIST_classifier(dense_net,lr,num_epochs,resize=(96, 96))

epoch 1, loss 0.428392            , train accuracy  0.864883, test accuracy 0.879100
epoch 2, loss 0.274547            , train accuracy  0.911417, test accuracy 0.896100
epoch 3, loss 0.235654            , train accuracy  0.923083, test accuracy 0.909800
epoch 4, loss 0.209992            , train accuracy  0.932317, test accuracy 0.918100
epoch 5, loss 0.190375            , train accuracy  0.939583, test accuracy 0.918000


En términos de conexiones entre capas, a diferencia de ResNet, donde las entradas y salidas se suman, DenseNet concatena entradas y salidas en la dimensión del canal. Aunque estas operaciones de concatenación reutilizan características para lograr eficiencia computacional, desafortunadamente conducen a un **alto consumo de memoria GPU.** Como resultado, la aplicación de DenseNet puede requerir implementaciones más eficientes en memoria que pueden aumentar el tiempo de entrenamiento.